# Publicar o Triunfo no Hugging Face

Executar no Colab, com CPU. Não treinar nem carregar modelos. O adaptador avaliado já está no Drive. A publicação terá destino `pedrojb04/triunfo`, público, com ficha, licença, configuração e resultados do teste.

Execute uma célula por vez. A primeira só prepara e verifica arquivos. A segunda autentica. A terceira envia e verifica os arquivos do commit publicado. Não colocar tokens no código, saídas compartilhadas ou mensagens do projeto.


## 1. Preparar o pacote no Drive

Verificar o ZIP de metadados e o SHA256 dos pesos avaliados. Copiar somente os arquivos necessários. Fixar a revisão na cópia do adapter_config, sem alterar os pesos nem o backup original. Nenhum envio nesta etapa.


In [ ]:
METADATA_ZIP_SHA256 = '6aa67dd518d355616f0ef4b2503d66c45dc9c6774ed4ccba693a6cda9e5ec259'

import hashlib
import json
import shutil
import zipfile
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive/Triunfo').is_dir():
    drive.mount('/content/drive')
raiz_publicacao = Path('/content/drive/MyDrive/Triunfo')
zip_metadados = raiz_publicacao / 'triunfo-hf-metadata-v1.zip'
assert hashlib.sha256(zip_metadados.read_bytes()).hexdigest() == METADATA_ZIP_SHA256

pasta_publicacao = raiz_publicacao / 'publication' / 'pedrojb04-triunfo'
pasta_publicacao.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_metadados) as z:
    meta = json.loads(z.read('_metadata_manifest.json'))
    assert set(z.namelist()) == set(meta['files']) | {'_metadata_manifest.json'}
    for nome, esperado in meta['files'].items():
        assert '/' not in nome and '\\' not in nome
        conteudo = z.read(nome)
        assert hashlib.sha256(conteudo).hexdigest() == esperado
        (pasta_publicacao / nome).write_bytes(conteudo)

config_publicacao = json.loads((pasta_publicacao / 'experiment_config.json').read_text())
avaliacao_publicacao = json.loads((pasta_publicacao / 'evaluation_results.json').read_text())
assert avaliacao_publicacao['single_call_counts']['finetuned']['call_exact'] == 280
assert avaliacao_publicacao['single_call_cases_scored'] == 444
origem_adapter = raiz_publicacao / 'runs/lora_epoch1_recovery/adapter_final'
assert hashlib.sha256((origem_adapter / 'adapter_model.safetensors').read_bytes()).hexdigest() == config_publicacao['adapter_sha256']

# Copia somente o adaptador e os arquivos necessários ao tokenizer.
permitidos = {
    'adapter_model.safetensors', 'adapter_config.json',
    'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json',
    'added_tokens.json', 'vocab.json', 'merges.txt', 'chat_template.jinja',
}
for arquivo in origem_adapter.rglob('*'):
    if not arquivo.is_file():
        continue
    relativo = arquivo.relative_to(origem_adapter)
    permitido = (len(relativo.parts) == 1 and relativo.name in permitidos) or (
        relativo.parts[0] == 'chat_templates' and arquivo.suffix == '.jinja'
    )
    if permitido:
        destino = pasta_publicacao / relativo
        destino.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(arquivo, destino)

config_adapter = json.loads((pasta_publicacao / 'adapter_config.json').read_text())
assert config_adapter['base_model_name_or_path'] == config_publicacao['model_id']
assert config_adapter['peft_type'] == 'LORA'
assert config_adapter['r'] == 16 and config_adapter['lora_alpha'] == 32
assert set(config_adapter['target_modules']) == {'q_proj', 'k_proj', 'v_proj', 'o_proj'}
# Fixa a revisão apenas na cópia de publicação; preserva o backup original.
config_adapter['revision'] = config_publicacao['model_revision']
config_adapter['inference_mode'] = True
(pasta_publicacao / 'adapter_config.json').write_text(
    json.dumps(config_adapter, ensure_ascii=False, indent=2) + '\n', encoding='utf-8',
)
assert hashlib.sha256((pasta_publicacao / 'adapter_model.safetensors').read_bytes()).hexdigest() == config_publicacao['adapter_sha256']

arquivos_publicacao = sorted(p for p in pasta_publicacao.rglob('*') if p.is_file() and p.name != 'publication_manifest.json')
for p in arquivos_publicacao:
    r = p.relative_to(pasta_publicacao)
    assert r.as_posix() in meta['files'] or r.name in permitidos or (r.parts[0] == 'chat_templates' and p.suffix == '.jinja')
manifesto_publicacao = {
    'repo_id': 'pedrojb04/triunfo',
    'repo_type': 'model',
    'public': True,
    'adapter_sha256': config_publicacao['adapter_sha256'],
    'model_revision': config_publicacao['model_revision'],
    'files': {p.relative_to(pasta_publicacao).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in arquivos_publicacao},
}
(pasta_publicacao / 'publication_manifest.json').write_text(
    json.dumps(manifesto_publicacao, ensure_ascii=False, indent=2) + '\n', encoding='utf-8',
)
print('Pacote de publicação preparado e conferido no Drive.')
print('Destino:', manifesto_publicacao['repo_id'])
print('Arquivos:', len(arquivos_publicacao) + 1)
print('Pesos idênticos aos avaliados:', manifesto_publicacao['adapter_sha256'])
print('Revisão do modelo original fixada:', manifesto_publicacao['model_revision'])
print('Nenhum modelo carregado, nenhuma geração e nenhum envio ao Hugging Face nesta célula.')


## 2. Autenticar no Hugging Face

Use o fluxo de login exibido pelo Colab. Se a versão da biblioteca pedir um token, use o campo privado e permissão de escrita para publicar. Aguarde a conclusão antes da próxima célula. Não compartilhe o token.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## 3. Publicar e conferir

Confere a conta pedrojb04 e os hashes locais. Não sobrescreve um repositório com arquivos de outro experimento. Envia o pacote e baixa os arquivos do commit apenas no Colab para confirmar seus hashes, sem executar o modelo.


In [ ]:
import hashlib
import json
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.errors import RepositoryNotFoundError

pasta_publicacao = Path('/content/drive/MyDrive/Triunfo/publication/pedrojb04-triunfo')
manifesto_publicacao = json.loads((pasta_publicacao / 'publication_manifest.json').read_text())
assert manifesto_publicacao['repo_id'] == 'pedrojb04/triunfo'
for nome, esperado in manifesto_publicacao['files'].items():
    assert hashlib.sha256((pasta_publicacao / nome).read_bytes()).hexdigest() == esperado
permitidos_upload = set(manifesto_publicacao['files']) | {'publication_manifest.json'}
assert {p.relative_to(pasta_publicacao).as_posix() for p in pasta_publicacao.rglob('*') if p.is_file()} == permitidos_upload

api_publicacao = HfApi()
conta_publicacao = api_publicacao.whoami()['name']
assert conta_publicacao == 'pedrojb04', 'Conta diferente do destino combinado. Não publicar.'
repo_publicacao = manifesto_publicacao['repo_id']
try:
    info_existente = api_publicacao.repo_info(repo_publicacao, repo_type='model')
except RepositoryNotFoundError:
    info_existente = None
if info_existente is not None:
    existentes = set(api_publicacao.list_repo_files(repo_publicacao, repo_type='model')) - {'.gitattributes'}
    if existentes:
        assert existentes <= permitidos_upload and 'publication_manifest.json' in existentes, (
            'O repositório já contém arquivos sem identificação deste experimento. Não sobrescrever.'
        )
        manifest_remoto = json.loads(Path(hf_hub_download(repo_publicacao, 'publication_manifest.json', repo_type='model')).read_text())
        assert manifest_remoto['adapter_sha256'] == manifesto_publicacao['adapter_sha256']
    assert not info_existente.private, 'Repositório existente privado: revisar a visibilidade antes de enviar.'

url_publicacao = api_publicacao.create_repo(repo_publicacao, repo_type='model', private=False, exist_ok=True)
commit_publicacao = api_publicacao.upload_folder(
    repo_id=repo_publicacao,
    repo_type='model',
    folder_path=str(pasta_publicacao),
    allow_patterns=sorted(permitidos_upload),
    commit_message='Publish Triunfo LoRA and held-out evaluation',
)
# Confere os arquivos no commit publicado; os downloads ficam só no Colab.
for nome in sorted(permitidos_upload):
    remoto = Path(hf_hub_download(repo_publicacao, nome, revision=commit_publicacao.oid, repo_type='model'))
    assert hashlib.sha256(remoto.read_bytes()).hexdigest() == hashlib.sha256((pasta_publicacao / nome).read_bytes()).hexdigest(), nome

registro_publicacao = {
    'repo_id': repo_publicacao,
    'url': str(url_publicacao),
    'commit': commit_publicacao.oid,
    'commit_url': commit_publicacao.commit_url,
    'remote_files_verified': len(permitidos_upload),
    'adapter_sha256': manifesto_publicacao['adapter_sha256'],
    'model_loaded': False,
}
(pasta_publicacao.parent / 'huggingface_publication.json').write_text(
    json.dumps(registro_publicacao, ensure_ascii=False, indent=2) + '\n', encoding='utf-8',
)
print(json.dumps(registro_publicacao, ensure_ascii=False, indent=2))
